# 🗳️ Election OCR Pipeline — Super AI Engineer Season 6 (Round 2)

**Thai Election Vote Tally OCR** — อ่านคะแนนเสียงจากเอกสารผลการเลือกตั้งไทย

---

## 📋 สารบัญ (Table of Contents)

1. [ติดตั้ง Dependencies](#section-1)
2. [ตั้งค่า API Key](#section-2)
3. [ดาวน์โหลดข้อมูล](#section-3)
4. [Data Models & Configuration](#section-4)
5. [Image Preprocessing](#section-5)
6. [Gemini OCR Client](#section-6)
7. [Prompt Templates](#section-7)
8. [Post-processing & Thai Numeral Handling](#section-8)
9. [Probabilistic Value Selection](#section-9)
10. [Digit-level Checksum Solver](#section-10)
11. [Document Discovery & Mapping](#section-11)
12. [Main Pipeline Orchestrator](#section-12)
13. [เลือกโหมดรัน](#section-13)
14. [Validation Mode (5 ตัวอย่าง)](#section-14)
15. [Full Run & สร้าง Submission](#section-15)

---

### แนวคิดหลัก (Core Approach)

- ใช้ **Gemini 2.5 Flash** อ่านภาพหลายรอบ (5 variants × ทุกหน้า + 1 all-pages call)
- **Probabilistic digit-level voting** — เลือกค่าที่ดีที่สุดจากหลาย readings
- **Checksum validation** — ใช้ผลรวม section 4.1 ตรวจสอบและแก้ไขค่าผิดพลาด
- **Digit confusion solver** — แก้ไขตัวเลขไทยที่คล้ายกัน (เช่น ๓↔๗, ๑↔๗)
- **Multi-variant preprocessing** — original, enhanced contrast, high contrast

<a name="section-1"></a>
## 1. ติดตั้ง Dependencies

In [ ]:
# ติดตั้ง libraries ที่จำเป็น
!pip install -q google-genai Pillow

<a name="section-2"></a>
## 2. ตั้งค่า API Key

ใส่ Gemini API Key (ได้จาก [Google AI Studio](https://aistudio.google.com/apikey))

In [ ]:
import os

# === ใส่ API Key ตรงนี้ ===
GEMINI_API_KEY = "BLANK_API_KEY"  # <-- เปลี่ยนเป็น Gemini API Key ของคุณ

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert GEMINI_API_KEY != "BLANK_API_KEY", "กรุณาใส่ Gemini API Key ก่อนรัน!"
print("✅ Gemini API Key ตั้งค่าเรียบร้อย")

<a name="section-3"></a>
## 3. ดาวน์โหลดข้อมูล

เชื่อมต่อ Google Drive แล้วอ่านข้อมูลจากโฟลเดอร์ที่แชร์ไว้

> ก่อนรัน: เปิด [โฟลเดอร์ข้อมูล](https://drive.google.com/drive/folders/1FkLP9hCqRtEuJSUIQp90r6xAXKD7LugB?usp=share_link) แล้วกด **"Add shortcut to Drive"** เพิ่มลง My Drive ของคุณก่อน

In [ ]:
from pathlib import Path

# === Mount Google Drive ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive เชื่อมต่อแล้ว")
except ImportError:
    print("ไม่ได้อยู่บน Colab — ใช้ path ใน local แทน")

# === หาโฟลเดอร์ข้อมูล ===
DATA_ROOT = None
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),   # Google Drive
    Path("/content/drive/Shareddrives"),
    Path("/content"),
    Path("."),
]

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("images"):
        if p.is_dir() and any(p.glob("*.png")):
            DATA_ROOT = p.parent
            print(f"✅ พบข้อมูลที่: {DATA_ROOT}")
            break
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "ไม่พบโฟลเดอร์ข้อมูล — กรุณา:\n"
        "1. เปิด https://drive.google.com/drive/folders/1FkLP9hCqRtEuJSUIQp90r6xAXKD7LugB\n"
        "2. คลิกขวา → Organize → Add shortcut → My Drive\n"
        "3. รันเซลล์นี้อีกครั้ง")

# === ตั้ง paths ===
IMAGE_DIR = DATA_ROOT / "images"
TEMPLATE_PATH = DATA_ROOT / "submission_template_v4.csv"
LABEL_DIR = DATA_ROOT / "sample_labels"

# ค้นหาใน subdirectories ถ้าไม่เจอตรง
if not IMAGE_DIR.exists():
    for p in DATA_ROOT.rglob("images"):
        if p.is_dir() and any(p.glob("*.png")):
            IMAGE_DIR = p; break
if not TEMPLATE_PATH.exists():
    for p in DATA_ROOT.rglob("submission_template_v4.csv"):
        TEMPLATE_PATH = p; break
if not LABEL_DIR.exists():
    for p in DATA_ROOT.rglob("sample_labels"):
        if p.is_dir(): LABEL_DIR = p; break

n_images = len(list(IMAGE_DIR.glob("*.png")))
print(f"\n📁 พบ {n_images} ภาพ")
print(f"   Images:   {IMAGE_DIR}")
print(f"   Template: {TEMPLATE_PATH}")
print(f"   Labels:   {LABEL_DIR}")
assert n_images > 0, f"ไม่พบภาพ .png ใน {IMAGE_DIR}"

<a name="section-4"></a>
## 4. Data Models & Configuration

กำหนดโครงสร้างข้อมูลและ configuration สำหรับ pipeline

In [ ]:
from __future__ import annotations
import csv
import hashlib
import json
import logging
import math
import re
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter

# === Data Models ===

class DocType(Enum):
    CONSTITUENCY = "constituency"
    PARTY_LIST = "party_list"

@dataclass
class DocumentInfo:
    doc_id: str
    doc_type: DocType
    province_code: str
    constituency: str
    image_paths: list[Path]
    expected_rows: int

@dataclass
class Pass1Row:
    number: int
    name: str | None
    party: str | None
    votes: str

@dataclass
class Pass1Result:
    rows: list[Pass1Row]
    section_4_1: int | None
    raw_response: str

@dataclass
class ReconciledRow:
    row_number: int
    vote_value: str
    confidence: float
    pass1_value: str | None
    pass2_value: str | None
    disagreement: bool

@dataclass
class DocumentResult:
    doc_id: str
    doc_type: DocType
    rows: list[ReconciledRow]
    section_4_1: int | None
    checksum_valid: bool | None
    checksum_diff: float | None
    warnings: list[str] = field(default_factory=list)

@dataclass
class ValidationResult:
    doc_id: str
    row_results: list[dict]
    doc_mean_distance: float

# === Configuration ===
CACHE_DIR = Path(".cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = Path("submission.csv")
RATE_LIMIT = 1.0
MAX_RETRIES = 3

# === Logging ===
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger("pipeline")

print("✅ Data models และ configuration พร้อมใช้งาน")

<a name="section-5"></a>
## 5. Image Preprocessing

เตรียมภาพหลาย variant เพื่อเพิ่มความแม่นยำในการอ่าน OCR
- **original**: ไม่แก้ไข
- **enhanced**: เพิ่ม contrast + sharpness (ดีสำหรับลายมือจาง)
- **high_contrast**: contrast สูงสำหรับตัวเลขที่อ่านยาก

In [ ]:
def preprocess_image(image_path: Path, variant: str = "original") -> Image.Image:
    """โหลดและ preprocess ภาพสำหรับ OCR"""
    img = Image.open(image_path)
    if variant == "original":
        return img
    if variant == "enhanced":
        img = ImageEnhance.Contrast(img).enhance(1.4)
        img = ImageEnhance.Sharpness(img).enhance(1.8)
        return img
    if variant == "high_contrast":
        img = ImageEnhance.Contrast(img).enhance(1.8)
        img = ImageEnhance.Sharpness(img).enhance(2.0)
        img = ImageEnhance.Brightness(img).enhance(1.1)
        return img
    return img

print("✅ Image preprocessing พร้อมใช้งาน")

<a name="section-6"></a>
## 6. Gemini OCR Client

ส่งภาพ + prompt ไปยัง Gemini API พร้อม retry logic และ rate limiting

In [ ]:
from google import genai
from google.genai import types

class GeminiOCR:
    def __init__(self, api_key: str, rate_limit: float = 1.0, max_retries: int = 3):
        self.client = genai.Client(api_key=api_key)
        self.model_flash = "gemini-2.5-flash"
        self.rate_limit = rate_limit
        self.max_retries = max_retries
        self._last_call = 0.0

    def _wait_rate_limit(self):
        elapsed = time.time() - self._last_call
        if elapsed < self.rate_limit:
            time.sleep(self.rate_limit - elapsed)

    def call_gemini(self, images: list, prompt: str, json_mode: bool = True,
                    temperature: float = 0.0) -> str | None:
        """ส่งภาพ + prompt ไป Gemini, return raw text หรือ None"""
        pil_images = []
        for img in images:
            if isinstance(img, (Path, str)):
                pil_images.append(Image.open(img))
            else:
                pil_images.append(img)
        contents = pil_images + [prompt]

        config_kwargs = {"temperature": temperature, "max_output_tokens": 65536}
        if json_mode:
            config_kwargs["response_mime_type"] = "application/json"
        config = types.GenerateContentConfig(**config_kwargs)

        for attempt in range(self.max_retries):
            try:
                self._wait_rate_limit()
                response = self.client.models.generate_content(
                    model=self.model_flash, contents=contents, config=config)
                self._last_call = time.time()
                text = response.text
                if text and text.strip():
                    return text
                logger.warning(f"Empty response (attempt {attempt+1})")
                continue
            except Exception as e:
                err_str = str(e)
                if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    wait = min(30, 2 ** (attempt + 2))
                    logger.warning(f"Rate limited, waiting {wait}s...")
                    time.sleep(wait)
                    continue
                wait = 2 ** attempt
                logger.warning(f"API error: {e}. Retrying in {wait}s...")
                time.sleep(wait)
        logger.error(f"Gemini API failed after {self.max_retries} retries")
        return None

# สร้าง OCR client
ocr = GeminiOCR(api_key=GEMINI_API_KEY, rate_limit=RATE_LIMIT, max_retries=MAX_RETRIES)
print("✅ Gemini OCR client พร้อมใช้งาน")

<a name="section-7"></a>
## 7. Prompt Templates

Prompt สำหรับอ่านเอกสาร 2 ประเภท: แบบแบ่งเขต (constituency) และแบบบัญชีรายชื่อ (party list)

In [ ]:
# === Prompt Templates ===
# ใช้ double braces {{ }} สำหรับ JSON ตัวอย่างใน prompt, single braces สำหรับ .format()

PROMPT_PARTY_LIST_JSON = """You are reading a Thai election party-list vote tally sheet (แบบ สส.6/2).

This page shows handwritten vote counts for political parties.
Table columns: ลำดับที่ (party number) | ชื่อพรรค (party name) | คะแนน (votes)

CRITICAL READING RULES:
1. The party number is PRINTED — read it exactly.
2. The vote count is HANDWRITTEN in Thai numerals.
3. Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9
4. FIRST count how many digits are in the number, THEN read each digit left-to-right.
5. Watch for confusable pairs:
   - ๑(1) is a single vertical stroke; ๗(7) has a horizontal top bar
   - ๓(3) has a flat/rounded top; ๗(7) has a pointed/angled top
   - ๖(6) has a loop at bottom-left; ๐(0) is a simple oval
   - ๕(5) has a hook shape; ๙(9) has a loop at top-right
   - ๘(8) has two stacked loops; ๓(3) has one curve
   - ๒(2) curves right then left; ๖(6) curves left with a bottom loop
6. Commas or dots between digits are thousand separators — ignore them.
7. A dash (—), blank cell, or illegible mark = 0.
8. If the last page shows section 4.1 total (รวมจำนวนคะแนนที่นับได้), read it too.

Return JSON:
{{
  "rows": [
    {{"number": 1, "votes": "346"}},
    {{"number": 2, "votes": "1313"}}
  ],
  "total": "98815"
}}

Rules:
- "number": the printed party number (integer)
- "votes": Arabic digits ONLY, no commas, as a string
- "total": section 4.1 total string, or null if not on this page
- Include EVERY row visible on this page, even if votes = "0"
"""

PROMPT_PARTY_LIST_JSON_ALL = """You are reading ALL pages of a Thai election party-list vote tally sheet (แบบ สส.6/2).
There are exactly 57 parties numbered 1 through 57.

Table columns: ลำดับที่ (party number) | ชื่อพรรค (party name) | คะแนน (votes)

CRITICAL READING RULES:
1. The party number is PRINTED — read it exactly.
2. The vote count is HANDWRITTEN in Thai numerals.
3. Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9
4. FIRST count how many digits are in the number, THEN read each digit left-to-right.
5. Watch for confusable pairs:
   - ๑(1) is a single vertical stroke; ๗(7) has a horizontal top bar
   - ๓(3) has a flat/rounded top; ๗(7) has a pointed/angled top
   - ๖(6) has a loop at bottom-left; ๐(0) is a simple oval
   - ๕(5) has a hook shape; ๙(9) has a loop at top-right
   - ๘(8) has two stacked loops; ๓(3) has one curve
   - ๒(2) curves right then left; ๖(6) curves left with a bottom loop
6. Commas or dots between digits are thousand separators — ignore them.
7. A dash (—), blank cell, or illegible mark = 0.
8. Read section 4.1 total (รวมจำนวนคะแนนที่นับได้) from the last page.

Return JSON:
{{
  "rows": [
    {{"number": 1, "votes": "346"}},
    {{"number": 2, "votes": "1313"}}
  ],
  "total": "98815"
}}

Rules:
- "number": printed party number (integer 1-57)
- "votes": Arabic digits ONLY, no commas, as a string
- "total": section 4.1 total string, or null
- You MUST return exactly 57 rows, one per party
"""

PROMPT_CONSTITUENCY_JSON = """You are reading a Thai election constituency vote tally sheet (แบบ สส.6/1).
This constituency has exactly {expected_rows} candidates.

The table is sorted by DESCENDING vote count (most votes at top).
Table columns: ลำดับที่ (rank) | หมายเลข (ballot number) | ชื่อผู้สมัคร (candidate name) | คะแนน (votes)

CRITICAL READING RULES:
1. RANK (column 1): sequential position from top, starting at 1.
2. BALLOT NUMBER (column 2): a small PRINTED Arabic numeral (1-{expected_rows}). Read this very carefully — it is NOT the same as the rank.
3. VOTES (column 4): HANDWRITTEN in Thai numerals.
4. Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9
5. FIRST count how many digits are in the vote number, THEN read each digit left-to-right.
6. Watch for confusable pairs:
   - ๑(1) is a single vertical stroke; ๗(7) has a horizontal top bar
   - ๓(3) has a flat/rounded top; ๗(7) has a pointed/angled top
   - ๖(6) has a loop at bottom-left; ๐(0) is a simple oval
   - ๕(5) has a hook shape; ๙(9) has a loop at top-right
   - ๘(8) has two stacked loops; ๓(3) has one curve
7. Commas or dots between digits are thousand separators — ignore them.
8. If section 4.1 total (รวมจำนวนคะแนนที่นับได้) is visible, read it too.

Return JSON:
{{
  "rows": [
    {{"rank": 1, "ballot": 5, "votes": "44405"}},
    {{"rank": 2, "ballot": 2, "votes": "25218"}}
  ],
  "total": "99026"
}}

Rules:
- "rank": row position from top (1 = highest votes), integer
- "ballot": the PRINTED ballot number from column 2, integer 1-{expected_rows}
- "votes": Arabic digits ONLY, no commas, as a string
- "total": section 4.1 total string, or null if not visible
- If this page has NO vote table: {{"rows": [], "total": null}}
"""

PROMPT_CONSTITUENCY_JSON_ALL = """You are reading ALL pages of a Thai election constituency vote tally sheet (แบบ สส.6/1).
Province {province_code}, constituency {constituency}. Exactly {expected_rows} candidates.

The table is sorted by DESCENDING vote count (most votes at top).
Table columns: ลำดับที่ (rank) | หมายเลข (ballot number) | ชื่อผู้สมัคร (candidate name) | คะแนน (votes)

CRITICAL READING RULES:
1. RANK (column 1): sequential position from top, starting at 1.
2. BALLOT NUMBER (column 2): a small PRINTED Arabic numeral (1-{expected_rows}). Read this very carefully — it is NOT the same as the rank.
3. VOTES (column 4): HANDWRITTEN in Thai numerals.
4. Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9
5. FIRST count how many digits are in the vote number, THEN read each digit left-to-right.
6. Watch for confusable pairs:
   - ๑(1) is a single vertical stroke; ๗(7) has a horizontal top bar
   - ๓(3) has a flat/rounded top; ๗(7) has a pointed/angled top
   - ๖(6) has a loop at bottom-left; ๐(0) is a simple oval
   - ๕(5) has a hook shape; ๙(9) has a loop at top-right
   - ๘(8) has two stacked loops; ๓(3) has one curve
7. Commas or dots between digits are thousand separators — ignore them.
8. Read section 4.1 total (รวมจำนวนคะแนนที่นับได้).

Return JSON:
{{
  "rows": [
    {{"rank": 1, "ballot": 5, "votes": "44405"}},
    {{"rank": 2, "ballot": 2, "votes": "25218"}}
  ],
  "total": "99026"
}}

Rules:
- "rank": row position from top (1 = highest votes), integer 1-{expected_rows}
- "ballot": the PRINTED ballot number from column 2, integer 1-{expected_rows}
- "votes": Arabic digits ONLY, no commas, as a string
- "total": section 4.1 total string, or null
- You MUST return exactly {expected_rows} rows
"""

def load_prompt(prompt_template: str, **kwargs) -> str:
    """เติมค่าลงใน prompt template"""
    return prompt_template.format(**kwargs)

def prompt_hash(prompt_text: str) -> str:
    """สร้าง hash สั้นๆ สำหรับ cache key"""
    return hashlib.md5(prompt_text.encode()).hexdigest()[:8]

print("✅ Prompt templates พร้อมใช้งาน")

<a name="section-8"></a>
## 8. Post-processing & Thai Numeral Handling

แปลงตัวเลขไทย → อารบิก, ทำความสะอาดค่า vote, ตรวจสอบ checksum

In [ ]:
# === Thai Numeral Conversion ===
THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

# คู่ตัวเลขไทยที่มักอ่านสับสน
DIGIT_CONFUSIONS = [
    ("3","7"),("7","3"),("1","7"),("7","1"),("6","2"),("2","6"),
    ("5","9"),("9","5"),("8","3"),("3","8"),("0","3"),("3","0"),
    ("1","4"),("4","1"),("6","0"),("0","6"),("8","9"),("9","8"),
    ("4","9"),("9","4"),("1","2"),("2","1"),("5","8"),("8","5"),
    ("0","9"),("9","0"),("4","7"),("7","4"),("4","5"),("5","4"),
    ("7","8"),("8","7"),
]

def convert_thai_numerals(text: str) -> str:
    return text.translate(THAI_DIGITS)

def clean_vote_value(raw: str) -> str:
    """ทำความสะอาดค่า vote ให้เป็นตัวเลขอารบิกล้วน"""
    if not raw or raw.strip() == "":
        return "0"
    text = convert_thai_numerals(raw)
    if "." in text:
        nums = re.findall(r"[\d.]+", text)
        if nums:
            try: return str(max(round(float(nums[0])), 0))
            except ValueError: pass
    digits = re.sub(r"[^\d]", "", text)
    if not digits:
        return "0"
    return digits.lstrip("0") or "0"

def validate_checksum(votes: list[int], section_4_1_total: int | None,
                      tolerance: float = 0.02) -> tuple[bool, dict]:
    """เปรียบเทียบผลรวม votes กับ section 4.1"""
    computed = sum(votes)
    if section_4_1_total is None or section_4_1_total == 0:
        return True, {"computed": computed, "expected": None, "diff_pct": None}
    diff = abs(computed - section_4_1_total)
    diff_pct = diff / section_4_1_total
    return diff_pct <= tolerance, {
        "computed": computed, "expected": section_4_1_total,
        "diff": diff, "diff_pct": round(diff_pct, 4)}

print("✅ Post-processing functions พร้อมใช้งาน")

<a name="section-9"></a>
## 9. Probabilistic Value Selection

เลือกค่าที่ดีที่สุดจากหลาย OCR readings โดยใช้ digit-level probability

In [ ]:
def build_digit_probs(candidates: list[str]) -> dict[str, float]:
    """สร้าง probability distribution จาก OCR readings หลายรอบ"""
    if not candidates:
        return {"0": 1.0}
    cleaned = [clean_vote_value(v) for v in candidates]
    non_zero = [c for c in cleaned if c != "0"]
    if non_zero:
        cleaned = non_zero
    # กรองค่าที่จำนวนหลักต่างกันมาก
    len_counter = Counter(len(v) for v in cleaned)
    lengths = sorted(len_counter.keys())
    if len(lengths) > 1:
        has_big_gap = any(lengths[i+1] - lengths[i] >= 2 for i in range(len(lengths)-1))
        if has_big_gap:
            len_groups = defaultdict(list)
            for v in cleaned:
                len_groups[len(v)].append(v)
            best_len = max(len_groups.keys(), key=lambda l: len(len_groups[l]))
            cleaned = len_groups[best_len]
    counter = Counter(cleaned)
    total = sum(counter.values())
    return {val: count / total for val, count in counter.items()}

def select_best_value(candidates: list[str]) -> tuple[str, float]:
    """เลือกค่าที่ดีที่สุดพร้อม confidence score"""
    probs = build_digit_probs(candidates)
    if not probs:
        return "0", 1.0
    best_val = max(probs, key=probs.get)
    best_conf = probs[best_val]
    if best_conf >= 0.6:
        return best_val, best_conf
    # Digit-by-digit majority voting
    cleaned = [clean_vote_value(v) for v in candidates]
    non_zero = [c for c in cleaned if c != "0"]
    if non_zero:
        cleaned = non_zero
    len_counter = Counter(len(v) for v in cleaned)
    lengths = sorted(len_counter.keys())
    if len(lengths) > 1:
        has_big_gap = any(lengths[i+1] - lengths[i] >= 2 for i in range(len(lengths)-1))
        if has_big_gap:
            len_groups = defaultdict(list)
            for v in cleaned:
                len_groups[len(v)].append(v)
            best_len = max(len_groups.keys(), key=lambda l: len(len_groups[l]))
            cleaned = len_groups[best_len]
    if not cleaned:
        return best_val, best_conf
    target_len = Counter(len(v) for v in cleaned).most_common(1)[0][0]
    same_len = [v for v in cleaned if len(v) == target_len]
    if len(same_len) >= 3 and target_len > 0:
        result_digits = []
        digit_conf = 1.0
        for pos in range(target_len):
            digits = [v[pos] for v in same_len]
            c = Counter(digits)
            result_digits.append(c.most_common(1)[0][0])
            digit_conf *= c.most_common(1)[0][1] / len(digits)
        return "".join(result_digits), digit_conf
    return best_val, best_conf

def _generate_digit_variants(value_str: str) -> list[tuple[str, int]]:
    """สร้าง variant จากการสับสนตัวเลข"""
    results = []
    for pos in range(len(value_str)):
        orig_digit = value_str[pos]
        for d_from, d_to in DIGIT_CONFUSIONS:
            if orig_digit == d_from:
                new_val = value_str[:pos] + d_to + value_str[pos + 1:]
                if len(new_val) > 1 and new_val[0] == "0":
                    continue
                results.append((new_val, pos))
    return results

def probabilistic_total_select(s41_values: list[int],
                                row_data: list[tuple[int, str, list[str]]]) -> int | None:
    """เลือก TOTAL ที่ดีที่สุดโดยใช้ vote sum เป็น cross-check"""
    if not s41_values:
        return None
    freq = Counter(s41_values)
    total_readings = sum(freq.values())
    top_val, top_count = freq.most_common(1)[0]
    if top_count >= total_readings * 0.6 and top_val > 0:
        return top_val
    vote_sum = 0
    for _, best_val, _ in row_data:
        try: vote_sum += int(best_val)
        except ValueError: pass
    expanded = {}
    for val, count in freq.items():
        if val == 0: continue
        expanded[val] = expanded.get(val, 0) + count
        for variant_str, _ in _generate_digit_variants(str(val)):
            variant = int(variant_str)
            if variant > 0:
                expanded[variant] = expanded.get(variant, 0) + count * 0.1
    if not expanded:
        return freq.most_common(1)[0][0] if freq else None
    best_val = None
    best_score = float('inf')
    for val, weight in expanded.items():
        dist = abs(val - vote_sum)
        score = dist / max(weight, 0.1)
        if score < best_score:
            best_score = score
            best_val = val
    return best_val

print("✅ Probabilistic value selection พร้อมใช้งาน")

<a name="section-10"></a>
## 10. Digit-level Checksum Solver

แก้ไขค่าผิดพลาดระดับตัวเลขเดียว โดยใช้ confusion probability ของตัวเลขไทย

In [ ]:
# Confusion probabilities จากการวิเคราะห์ข้อมูลจริง
_RAW_CONFUSION_COUNTS = {
    ("1","3"):1362,("4","8"):778,("1","2"):613,("2","3"):594,("2","6"):542,
    ("3","7"):538,("4","5"):534,("2","4"):484,("3","4"):474,("3","8"):390,
    ("5","8"):386,("1","4"):380,("1","7"):380,("3","6"):348,("1","8"):339,
    ("5","7"):308,("9","7"):296,("9","4"):280,("9","8"):268,("1","6"):260,
    ("5","6"):240,("6","7"):230,("6","8"):220,("6","9"):210,("0","3"):200,
    ("0","8"):180,("0","6"):170,("0","9"):160,("0","1"):150,("0","2"):140,
    ("0","4"):130,("0","5"):120,("0","7"):110,("7","8"):200,("5","9"):190,
    ("2","7"):180,("2","8"):170,("4","6"):160,("4","7"):150,
}

CONFUSION_PROB = defaultdict(dict)
_max_count = max(_RAW_CONFUSION_COUNTS.values())
for (a, b), count in _RAW_CONFUSION_COUNTS.items():
    w = count / _max_count
    CONFUSION_PROB[a][b] = max(CONFUSION_PROB[a].get(b, 0), w)
    CONFUSION_PROB[b][a] = max(CONFUSION_PROB[b].get(a, 0), w)

def _digit_change_score(old_digit: str, new_digit: str) -> float:
    if old_digit == new_digit: return 0.0
    return CONFUSION_PROB.get(old_digit, {}).get(new_digit, 0.05)

def solve_checksum_digit(values: dict[int, str], target_sum: int,
                          expected_numbers: list[int],
                          candidates: dict[int, list[str]] | None = None,
                          max_changes: int = 2) -> dict[int, str] | None:
    """หาการแก้ไขระดับตัวเลขที่ทำให้ checksum ถูกต้อง"""
    current_sum = sum(int(values.get(n, "0")) for n in expected_numbers)
    diff = target_sum - current_sum
    if diff == 0: return None
    abs_diff = abs(diff)
    if target_sum > 0 and abs_diff / target_sum > 0.10: return None
    cands = candidates or {}

    # Phase 1: single-digit change ที่มี model candidate สนับสนุน
    best_supported = None
    for n in expected_numbers:
        cur = values.get(n, "0")
        if not cur.isdigit(): continue
        for pos in range(len(cur)):
            place = 10 ** (len(cur) - 1 - pos)
            old_d = cur[pos]
            for new_d_int in range(10):
                new_d = str(new_d_int)
                if new_d == old_d: continue
                delta = (new_d_int - int(old_d)) * place
                if delta != diff: continue
                new_val = cur[:pos] + new_d + cur[pos + 1:]
                if len(new_val) > 1 and new_val[0] == "0": continue
                conf_score = _digit_change_score(old_d, new_d)
                supported = new_val in cands.get(n, [])
                score = conf_score + (1.0 if supported else 0.0)
                if best_supported is None or score > best_supported[0]:
                    best_supported = (score, n, new_val)

    if best_supported and best_supported[0] >= 0.3:
        _, n, new_val = best_supported
        result = dict(values)
        result[n] = new_val
        return result

    if max_changes < 2: return None

    # Phase 2: two single-digit changes
    row_deltas = {}
    for n in expected_numbers:
        cur = values.get(n, "0")
        if not cur.isdigit(): continue
        deltas = []
        for pos in range(len(cur)):
            place = 10 ** (len(cur) - 1 - pos)
            old_d = cur[pos]
            for new_d_int in range(10):
                new_d = str(new_d_int)
                if new_d == old_d: continue
                delta = (new_d_int - int(old_d)) * place
                new_val = cur[:pos] + new_d + cur[pos + 1:]
                if len(new_val) > 1 and new_val[0] == "0": continue
                conf_score = _digit_change_score(old_d, new_d)
                supported = new_val in cands.get(n, [])
                score = conf_score + (1.0 if supported else 0.0)
                if score >= 0.2:
                    deltas.append((delta, new_val, score))
        if deltas: row_deltas[n] = deltas

    delta_index = defaultdict(list)
    for n, deltas in row_deltas.items():
        for delta, new_val, score in deltas:
            delta_index[delta].append((n, new_val, score))

    best_double = None
    for n_a, deltas_a in row_deltas.items():
        for delta_a, val_a, score_a in deltas_a:
            needed_b = diff - delta_a
            if needed_b == 0: continue
            for n_b, val_b, score_b in delta_index.get(needed_b, []):
                if n_b <= n_a: continue
                combined = (score_a + score_b) / 2
                if best_double is None or combined > best_double[0]:
                    best_double = (combined, [(n_a, val_a), (n_b, val_b)])

    if best_double and best_double[0] >= 1.0:
        result = dict(values)
        for n, val in best_double[1]:
            result[n] = val
        return result
    return None

print("✅ Digit-level checksum solver พร้อมใช้งาน")

<a name="section-11"></a>
## 11. Document Discovery & Mapping

ค้นหาเอกสารจากไฟล์ภาพ, จัดกลุ่มตาม doc_id, และ map ผลลัพธ์กลับไปยัง submission template

In [ ]:
FILENAME_RE = re.compile(
    r"^(constituency|party_list)_(\d+)_(\d+?)(?:_page(\d+))?\.png$"
)

def parse_filename(filename: str):
    m = FILENAME_RE.match(filename)
    if not m: raise ValueError(f"Cannot parse: {filename}")
    doc_type, province, constituency, page = m.groups()
    return doc_type, province, constituency, int(page) if page else None

def count_expected_rows(template_path: Path) -> dict[str, int]:
    counts = defaultdict(int)
    with open(template_path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            doc_id = row["id"].rsplit("_", 1)[0]
            counts[doc_id] += 1
    return dict(counts)

def discover_documents(image_dir: Path, template_path: Path) -> list[DocumentInfo]:
    """สแกนภาพ, จัดกลุ่มตามเอกสาร, cross-reference กับ template"""
    expected = count_expected_rows(template_path)
    groups = defaultdict(list)
    for img in sorted(image_dir.glob("*.png")):
        try:
            doc_type, province, constituency, page = parse_filename(img.name)
        except ValueError:
            continue
        doc_id = f"{doc_type}_{province}_{constituency}"
        groups[doc_id].append((page if page else 1, img))

    documents = []
    for doc_id, pages in sorted(groups.items()):
        pages.sort(key=lambda x: x[0])
        image_paths = [p for _, p in pages]
        if doc_id.startswith("party_list"):
            doc_type = DocType.PARTY_LIST
            prefix = "party_list"
        else:
            doc_type = DocType.CONSTITUENCY
            prefix = "constituency"
        remainder = doc_id[len(prefix) + 1:]
        parts = remainder.split("_", 1)
        province_code = parts[0]
        constituency = parts[1] if len(parts) > 1 else ""
        documents.append(DocumentInfo(
            doc_id=doc_id, doc_type=doc_type, province_code=province_code,
            constituency=constituency, image_paths=image_paths,
            expected_rows=expected.get(doc_id, 0)))
    logger.info(f"พบ {len(documents)} เอกสาร ({sum(len(d.image_paths) for d in documents)} ภาพ)")
    return documents

def load_submission_template(template_path: Path) -> dict[str, list[str]]:
    groups = defaultdict(list)
    with open(template_path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            row_id = row["id"]
            doc_id = row_id.rsplit("_", 1)[0]
            groups[doc_id].append(row_id)
    return dict(groups)

def map_rows_to_template(doc_id: str, reconciled_rows: list[ReconciledRow],
                          template_row_ids: list[str]) -> dict[str, str]:
    result = {}
    sorted_rows = sorted(reconciled_rows, key=lambda r: r.row_number)
    expected = len(template_row_ids)
    if len(sorted_rows) > expected:
        sorted_rows = sorted_rows[:expected]
    for i, row_id in enumerate(template_row_ids):
        result[row_id] = sorted_rows[i].vote_value if i < len(sorted_rows) else "0"
    return result

def write_submission_csv(results: dict[str, str], template_path: Path, output_path: Path):
    with open(template_path, encoding="utf-8") as f:
        template_ids = [row["id"] for row in csv.DictReader(f)]
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "votes"])
        for row_id in template_ids:
            writer.writerow([row_id, results.get(row_id, "0")])

def _get_expected_numbers(template_map, doc_id):
    row_ids = template_map.get(doc_id, [])
    return sorted(int(rid.rsplit("_", 1)[1]) for rid in row_ids)

print("✅ Document discovery & mapping พร้อมใช้งาน")

<a name="section-12"></a>
## 12. Main Pipeline Orchestrator

ฟังก์ชันหลักสำหรับประมวลผลเอกสารแต่ละฉบับ:
1. **Party List**: 5 calls/page (3 variants × 2 temperatures) + 1 all-pages call
2. **Constituency**: 5 calls/page + 1 all-pages call, rank-based extraction
3. **Post-processing**: probabilistic selection → checksum validation → digit solver

In [ ]:
# === JSON Parsing ===
def _parse_json_votes(raw: str) -> tuple[list[dict], str | None]:
    """Parse JSON response จาก Gemini"""
    if not raw or not raw.strip(): return [], None
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = raw.translate(thai_map)
    text = re.sub(r'"(\d+),(\d+),(\d+)"', lambda m: f'"{m.group(1)}{m.group(2)}{m.group(3)}"', text)
    text = re.sub(r'"(\d+),(\d+)"', lambda m: f'"{m.group(1)}{m.group(2)}"', text)
    data = None
    try: data = json.loads(text)
    except: pass
    if data is None:
        m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
        if m:
            try: data = json.loads(m.group(1))
            except: pass
    if data is None:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            try: data = json.loads(text[start:end + 1])
            except: pass
    if data is None: return [], None
    if isinstance(data, list): data = {"rows": data, "total": None}
    rows = data.get("rows", [])
    total_str = data.get("total")
    if total_str is not None:
        total_str = re.sub(r"[^\d]", "", str(total_str))
        if not total_str: total_str = None
    return rows, total_str

# === Cache ===
class ResultCache:
    def __init__(self, cache_dir: Path):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    def _path(self, doc_id, pass_name): return self.cache_dir / f"{doc_id}_{pass_name}.json"
    def has_result(self, doc_id, pass_name): return self._path(doc_id, pass_name).exists()
    def save_result(self, doc_id, pass_name, data, prompt_hash=""):
        from datetime import datetime
        payload = {"doc_id": doc_id, "pass_name": pass_name,
                   "timestamp": datetime.now().isoformat(), "prompt_hash": prompt_hash, **data}
        self._path(doc_id, pass_name).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    def load_result(self, doc_id, pass_name):
        p = self._path(doc_id, pass_name)
        if not p.exists(): return None
        return json.loads(p.read_text(encoding="utf-8"))

cache = ResultCache(CACHE_DIR)

print("✅ JSON parsing & cache พร้อมใช้งาน")

### 12.1 Party List Processing

ประมวลผลเอกสารแบบบัญชีรายชื่อ — 5 calls/page + 1 all-pages call

In [ ]:
def _process_party_list(doc, expected_numbers):
    """ประมวลผล party list: 5 calls/page + 1 all-pages"""
    prompt_kwargs = {"province_code": doc.province_code,
                     "constituency": doc.constituency, "expected_rows": doc.expected_rows}
    prompt_text = load_prompt(PROMPT_PARTY_LIST_JSON, **prompt_kwargs)
    p_hash = prompt_hash(prompt_text)
    full_cache_key = f"json_v3_party_list_{p_hash}"

    # ตรวจ cache
    cached = cache.load_result(doc.doc_id, full_cache_key)
    if cached and cached.get("prompt_hash") == p_hash:
        logger.info(f"  ใช้ cached result สำหรับ {doc.doc_id}")
        rows = [Pass1Row(r["number"], None, None, r["votes"]) for r in cached.get("rows", [])]
        vc = cached.get("vote_candidates")
        vote_cands = {int(k): v for k, v in vc.items()} if vc else None
        return Pass1Result(rows=rows, section_4_1=cached.get("section_4_1"), raw_response="cached"), vote_cands

    all_votes = defaultdict(list)
    s41_values = []
    raw_responses = []

    # 5 calls/page: 3 variants × 2 temperatures
    calls_per_page = [("original", 0.0), ("enhanced", 0.0), ("high_contrast", 0.0),
                      ("original", 0.1), ("enhanced", 0.1)]
    total_api = len(doc.image_paths) * len(calls_per_page) + 1
    api_count = 0

    for page_path in doc.image_paths:
        for variant, temp in calls_per_page:
            api_count += 1
            print(f"\r  API [{api_count}/{total_api}] {page_path.stem}/{variant}", end="")
            img = preprocess_image(page_path, variant)
            raw = ocr.call_gemini([img], prompt_text, json_mode=True, temperature=temp)
            if raw is None: continue
            raw_responses.append({"page": page_path.name, "variant": variant, "raw": raw})
            rows_data, total_str = _parse_json_votes(raw)
            for r in rows_data:
                try: num = int(r.get("number", 0))
                except: continue
                if 1 <= num <= 57:
                    cleaned = clean_vote_value(str(r.get("votes", "0")))
                    if cleaned != "0" or not all_votes.get(num):
                        all_votes[num].append(cleaned)
            if total_str:
                try: s41_values.append(int(total_str))
                except: pass

    # All-pages call
    all_prompt = load_prompt(PROMPT_PARTY_LIST_JSON_ALL, **prompt_kwargs)
    api_count += 1
    print(f"\r  API [{api_count}/{total_api}] all-pages        ", end="")
    raw = ocr.call_gemini(doc.image_paths, all_prompt, json_mode=True, temperature=0.0)
    if raw:
        raw_responses.append({"page": "all", "raw": raw})
        rows_data, total_str = _parse_json_votes(raw)
        # ตรวจ garbage response
        ap_values = [clean_vote_value(str(r.get("votes", "0"))) for r in rows_data]
        ap_non_zero = [v for v in ap_values if v != "0"]
        is_garbage = False
        if ap_non_zero:
            val_freq = Counter(ap_non_zero)
            if val_freq.most_common(1)[0][1] > len(expected_numbers) * 0.2:
                is_garbage = True
            if len(Counter(len(v) for v in ap_non_zero)) == 1 and len(ap_non_zero) >= 20:
                is_garbage = True
        if not is_garbage:
            for r in rows_data:
                try: num = int(r.get("number", 0))
                except: continue
                if 1 <= num <= 57:
                    cleaned = clean_vote_value(str(r.get("votes", "0")))
                    if cleaned != "0": all_votes[num].append(cleaned)
            if total_str:
                try: s41_values.append(int(total_str))
                except: pass
    print()

    # เลือกค่าที่ดีที่สุด
    row_data_for_total = []
    rows = []
    for num in expected_numbers:
        candidates = all_votes.get(num, [])
        best, conf = select_best_value(candidates) if candidates else ("0", 1.0)
        rows.append(Pass1Row(number=num, name=None, party=None, votes=best))
        row_data_for_total.append((num, best, candidates))

    section_4_1 = probabilistic_total_select(s41_values, row_data_for_total)

    # บันทึก cache
    vc_for_cache = {str(k): v for k, v in all_votes.items()}
    cache.save_result(doc.doc_id, full_cache_key, {
        "rows": [{"number": r.number, "votes": r.votes} for r in rows],
        "section_4_1": section_4_1, "vote_candidates": vc_for_cache,
        "s41_values": s41_values, "raw_responses": raw_responses,
    }, prompt_hash=p_hash)

    return Pass1Result(rows=rows, section_4_1=section_4_1, raw_response="aggregated"), dict(all_votes)

print("✅ Party list processing พร้อมใช้งาน")

### 12.2 Constituency Processing

ประมวลผลเอกสารแบบแบ่งเขต — rank-based extraction พร้อม ballot mapping

In [ ]:
def _validate_constituency_allpages(rows_data, rank_votes, num_candidates):
    """ตรวจว่า all-pages response สอดคล้องกับ per-page readings"""
    if not rank_votes: return True
    pp_rank1 = rank_votes.get(1, [])
    if not pp_rank1: return True
    ap_rank1_vote = None
    for r in rows_data:
        try: rank = int(r.get("rank", 0))
        except: continue
        if rank == 1:
            ap_rank1_vote = clean_vote_value(str(r.get("votes", "0")))
            break
    if not ap_rank1_vote or ap_rank1_vote == "0": return True
    pp_cleaned = [clean_vote_value(v) for v in pp_rank1]
    pp_non_zero = [int(v) for v in pp_cleaned if v != "0" and v.isdigit()]
    if not pp_non_zero: return True
    pp_median = sorted(pp_non_zero)[len(pp_non_zero) // 2]
    try: ap_val = int(ap_rank1_vote)
    except: return True
    if pp_median > 0:
        ratio = ap_val / pp_median
        if ratio > 2.0 or ratio < 0.5: return False
    if abs(len(str(pp_median)) - len(ap_rank1_vote)) >= 2: return False
    return True

def _process_constituency(doc, expected_numbers):
    """ประมวลผล constituency: rank-based extraction"""
    max_num = max(expected_numbers)
    num_candidates = len(expected_numbers)
    prompt_kwargs = {"province_code": doc.province_code,
                     "constituency": doc.constituency, "expected_rows": doc.expected_rows}
    prompt_text = load_prompt(PROMPT_CONSTITUENCY_JSON, **prompt_kwargs)
    p_hash = prompt_hash(prompt_text)
    full_cache_key = f"json_v3_constituency_{p_hash}"

    cached = cache.load_result(doc.doc_id, full_cache_key)
    if cached and cached.get("prompt_hash") == p_hash:
        logger.info(f"  ใช้ cached result สำหรับ {doc.doc_id}")
        rows = [Pass1Row(r["number"], None, None, r["votes"]) for r in cached.get("rows", [])]
        vc = cached.get("vote_candidates")
        vote_cands = {int(k): v for k, v in vc.items()} if vc else None
        return Pass1Result(rows=rows, section_4_1=cached.get("section_4_1"), raw_response="cached"), vote_cands

    rank_votes = defaultdict(list)
    rank_to_ballot = defaultdict(list)
    s41_values = []
    raw_responses = []

    # ข้ามหน้าปก สำหรับเอกสาร 3+ หน้า
    ocr_pages = doc.image_paths
    if len(doc.image_paths) >= 3:
        table_pages = [p for p in doc.image_paths if "_page" in p.stem]
        if len(table_pages) >= 2:
            ocr_pages = table_pages
            logger.info(f"  ข้ามหน้าปก, ใช้ {len(table_pages)} หน้าตาราง")

    calls_per_page = [("original", 0.0), ("enhanced", 0.0), ("high_contrast", 0.0),
                      ("original", 0.1), ("enhanced", 0.1)]
    total_api = len(ocr_pages) * len(calls_per_page) + 1
    api_count = 0

    for page_path in ocr_pages:
        for variant, temp in calls_per_page:
            api_count += 1
            print(f"\r  API [{api_count}/{total_api}] {page_path.stem}/{variant}", end="")
            img = preprocess_image(page_path, variant)
            raw = ocr.call_gemini([img], prompt_text, json_mode=True, temperature=temp)
            if raw is None: continue
            raw_responses.append({"page": page_path.name, "variant": variant, "raw": raw})
            rows_data, total_str = _parse_json_votes(raw)
            for r in rows_data:
                try:
                    rank = int(r.get("rank", 0))
                    ballot = int(r.get("ballot", r.get("ballot_number", 0)))
                except: continue
                if 1 <= rank <= num_candidates:
                    cleaned = clean_vote_value(str(r.get("votes", "0")))
                    if cleaned != "0": rank_votes[rank].append(cleaned)
                    if 1 <= ballot <= max_num:
                        rank_to_ballot[rank].append(ballot)
            if total_str:
                try: s41_values.append(int(total_str))
                except: pass

    # All-pages call
    all_prompt = load_prompt(PROMPT_CONSTITUENCY_JSON_ALL, **prompt_kwargs)
    api_count += 1
    print(f"\r  API [{api_count}/{total_api}] all-pages        ", end="")
    raw = ocr.call_gemini(doc.image_paths, all_prompt, json_mode=True, temperature=0.0)
    if raw:
        raw_responses.append({"page": "all", "raw": raw})
        rows_data, total_str = _parse_json_votes(raw)
        if _validate_constituency_allpages(rows_data, rank_votes, num_candidates):
            for r in rows_data:
                try:
                    rank = int(r.get("rank", 0))
                    ballot = int(r.get("ballot", r.get("ballot_number", 0)))
                except: continue
                if 1 <= rank <= num_candidates:
                    cleaned = clean_vote_value(str(r.get("votes", "0")))
                    if cleaned != "0": rank_votes[rank].append(cleaned)
                    if 1 <= ballot <= max_num:
                        rank_to_ballot[rank].append(ballot)
            if total_str:
                try: s41_values.append(int(total_str))
                except: pass
    print()

    # เลือกค่าที่ดีที่สุดต่อ rank
    rank_final_votes = {}
    row_data_for_total = []
    for rank in range(1, num_candidates + 1):
        candidates = rank_votes.get(rank, [])
        best, conf = select_best_value(candidates) if candidates else ("0", 1.0)
        rank_final_votes[rank] = best
        row_data_for_total.append((rank, best, candidates))

    section_4_1 = probabilistic_total_select(s41_values, row_data_for_total)

    rows = []
    rank_vote_candidates = {}
    for rank in range(1, num_candidates + 1):
        rows.append(Pass1Row(number=rank, name=None, party=None, votes=rank_final_votes.get(rank, "0")))
        if rank in rank_votes: rank_vote_candidates[rank] = rank_votes[rank]

    # บันทึก cache
    vc_for_cache = {str(k): v for k, v in rank_vote_candidates.items()}
    cache.save_result(doc.doc_id, full_cache_key, {
        "rows": [{"number": r.number, "votes": r.votes} for r in rows],
        "section_4_1": section_4_1, "vote_candidates": vc_for_cache,
        "s41_values": s41_values, "raw_responses": raw_responses,
    }, prompt_hash=p_hash)

    return Pass1Result(rows=rows, section_4_1=section_4_1, raw_response="aggregated"), rank_vote_candidates

print("✅ Constituency processing พร้อมใช้งาน")

### 12.3 Document Processing & Validation

ฟังก์ชันหลักสำหรับประมวลผลเอกสาร + validation ด้วย Levenshtein distance

In [ ]:
def process_document(doc, expected_numbers):
    """ประมวลผลเอกสาร 1 ฉบับ"""
    doc_type_label = "CONST" if doc.doc_type == DocType.CONSTITUENCY else "PARTY"
    logger.info(f"{doc_type_label} {doc.doc_id} — {len(doc.image_paths)} pages, {len(expected_numbers)} rows")

    if doc.doc_type == DocType.CONSTITUENCY:
        pass1, vote_candidates = _process_constituency(doc, expected_numbers)
    else:
        pass1, vote_candidates = _process_party_list(doc, expected_numbers)

    if pass1 is None:
        logger.error(f"  OCR failed สำหรับ {doc.doc_id}")
        rows = [ReconciledRow(n, "0", 0.0, None, None, False) for n in expected_numbers]
        return DocumentResult(doc.doc_id, doc.doc_type, rows, None, None, None, ["OCR failed"])

    # สร้าง vote map
    vote_map = {r.number: clean_vote_value(r.votes) for r in pass1.rows}
    section_4_1 = pass1.section_4_1
    ensemble_values = dict(vote_map)

    # Digit solver
    if section_4_1 and section_4_1 > 0:
        merged_cands = {}
        for n in expected_numbers:
            c = list((vote_candidates or {}).get(n, []))
            merged_cands[n] = c
        current_sum = sum(int(ensemble_values.get(n, "0")) for n in expected_numbers)
        if current_sum != section_4_1:
            digit_fix = solve_checksum_digit(ensemble_values, section_4_1, expected_numbers,
                                              candidates=merged_cands, max_changes=2)
            if digit_fix:
                changes = sum(1 for n in expected_numbers if digit_fix.get(n) != ensemble_values.get(n))
                if changes > 0:
                    logger.info(f"  DigitSolver แก้ไข {changes} ค่า")
                    ensemble_values = digit_fix

    # สร้าง reconciled rows
    reconciled = []
    for n in expected_numbers:
        final_val = ensemble_values.get(n, "0")
        gemini_val = vote_map.get(n, "0")
        reconciled.append(ReconciledRow(
            row_number=n, vote_value=final_val, confidence=1.0,
            pass1_value=gemini_val, pass2_value=None,
            disagreement=(final_val != gemini_val)))

    # ตรวจ checksum
    vote_ints = [int(r.vote_value) for r in reconciled]
    checksum_valid, checksum_info = validate_checksum(vote_ints, section_4_1)
    if checksum_valid:
        logger.info(f"  ✓ Checksum OK — sum={checksum_info['computed']} expected={checksum_info['expected']}")
    else:
        logger.warning(f"  ✗ Checksum FAIL — sum={checksum_info['computed']} expected={checksum_info['expected']}")

    return DocumentResult(
        doc_id=doc.doc_id, doc_type=doc.doc_type, rows=reconciled,
        section_4_1=section_4_1, checksum_valid=checksum_valid,
        checksum_diff=checksum_info.get("diff_pct"), warnings=[])

# === Validation ===
def levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2): return levenshtein_distance(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[j] + 1, prev[j + 1] + 1, prev[j] + (0 if c1 == c2 else 1)))
        prev = curr
    return prev[-1]

def validate_against_labels(predictions: dict[str, str], label_dir: Path):
    """เปรียบเทียบ predictions กับ sample labels"""
    results = []
    all_distances = []
    for label_file in sorted(label_dir.glob("*.json")):
        with open(label_file, encoding="utf-8") as f:
            label = json.load(f)
        doc_type = label["type"]
        province = label["province_code"]
        constituency = label["constituency_number"]
        doc_id = f"{doc_type}_{province}_{constituency}"
        row_results = []
        if doc_type == "constituency":
            sorted_by_votes = sorted(label["results"], key=lambda x: -x["votes"])
            for rank, entry in enumerate(sorted_by_votes, start=1):
                actual = str(entry["votes"])
                row_id = f"{doc_id}_{rank}"
                predicted = predictions.get(row_id, "0")
                dist = levenshtein_distance(predicted, actual)
                all_distances.append(dist)
                row_results.append({"row_id": row_id, "predicted": predicted, "actual": actual, "distance": dist})
        else:
            for entry in label["results"]:
                row_id = f"{doc_id}_{entry['number']}"
                actual = str(entry["votes"])
                predicted = predictions.get(row_id, "0")
                dist = levenshtein_distance(predicted, actual)
                all_distances.append(dist)
                row_results.append({"row_id": row_id, "predicted": predicted, "actual": actual, "distance": dist})
        doc_mean = sum(r["distance"] for r in row_results) / max(len(row_results), 1)
        results.append(ValidationResult(doc_id=doc_id, row_results=row_results, doc_mean_distance=doc_mean))
    overall_mld = sum(all_distances) / max(len(all_distances), 1)
    return results, overall_mld

print("✅ Document processing & validation พร้อมใช้งาน")

<a name="section-13"></a>
## 13. เลือกโหมดรัน

ตั้งค่า `VALIDATION_MODE = True` เพื่อรันเฉพาะ 5 ตัวอย่างที่มี labels สำหรับตรวจสอบ
หรือ `False` เพื่อรันทั้งหมด 300 เอกสาร

In [ ]:
# === ตั้งค่าโหมดรัน ===
VALIDATION_MODE = True  # True = รัน 5 ตัวอย่าง, False = รันทั้งหมด

# 5 ตัวอย่างสำหรับ validation (มี labels ให้ตรวจสอบ)
VALIDATION_DOC_IDS = {
    "constituency_10_1",
    "constituency_12_6",
    "party_list_10_2",
    "party_list_16_4",
    "party_list_33_3",
}

print(f"โหมด: {'VALIDATION (5 ตัวอย่าง)' if VALIDATION_MODE else 'FULL RUN (300 เอกสาร)'}")

<a name="section-14"></a>
## 14. รัน Pipeline

ประมวลผลเอกสารทั้งหมดตามโหมดที่เลือก

In [ ]:
# === รัน Pipeline ===
template_map = load_submission_template(TEMPLATE_PATH)
documents = discover_documents(IMAGE_DIR, TEMPLATE_PATH)

if VALIDATION_MODE:
    documents = [d for d in documents if d.doc_id in VALIDATION_DOC_IDS]

total_docs = len(documents)
print(f"{'='*60}")
print(f"  {'VALIDATION MODE' if VALIDATION_MODE else 'FULL RUN'} — {total_docs} เอกสาร, "
      f"{sum(len(d.image_paths) for d in documents)} ภาพ")
print(f"{'='*60}")

all_predictions = {}
pipeline_start = time.time()

for doc_idx, doc in enumerate(documents, 1):
    expected_numbers = _get_expected_numbers(template_map, doc.doc_id)
    if not expected_numbers:
        logger.warning(f"ไม่พบ expected numbers สำหรับ {doc.doc_id}")
        continue

    doc_start = time.time()
    print(f"\n[{doc_idx}/{total_docs}]")
    result = process_document(doc, expected_numbers)
    elapsed = time.time() - doc_start

    template_row_ids = template_map.get(doc.doc_id, [])
    mapped = map_rows_to_template(doc.doc_id, result.rows, template_row_ids)
    all_predictions.update(mapped)
    print(f"  {len(result.rows)} rows, {elapsed:.1f}s")

total_elapsed = time.time() - pipeline_start
print(f"\n{'='*60}")
print(f"  เสร็จสิ้น! {len(all_predictions)} predictions, {total_elapsed:.0f}s")
print(f"{'='*60}")

### ตรวจสอบผลลัพธ์ (Validation)

เปรียบเทียบ predictions กับ sample labels ด้วย Mean Levenshtein Distance

In [ ]:
if VALIDATION_MODE:
    results, overall_mld = validate_against_labels(all_predictions, LABEL_DIR)
    print(f"\n{'='*60}")
    print(f"  Overall Mean Levenshtein Distance: {overall_mld:.4f}")
    print(f"{'='*60}")
    for vr in results:
        errors = [r for r in vr.row_results if r["distance"] > 0]
        total_rows = len(vr.row_results)
        correct = total_rows - len(errors)
        status = "✓" if len(errors) == 0 else "✗"
        print(f"  {status} {vr.doc_id} — MLD={vr.doc_mean_distance:.4f}, {correct}/{total_rows} ถูกต้อง")
        for r in errors:
            print(f"    {r['row_id']}: predicted={r['predicted']} actual={r['actual']} dist={r['distance']}")
else:
    print("ข้าม validation (ไม่ได้อยู่ใน validation mode)")

<a name="section-15"></a>
## 15. สร้างไฟล์ Submission

บันทึกผลลัพธ์เป็น CSV สำหรับส่ง Kaggle

In [ ]:
# สร้างไฟล์ submission
write_submission_csv(all_predictions, TEMPLATE_PATH, OUTPUT_PATH)
print(f"✅ บันทึก submission ที่: {OUTPUT_PATH}")
print(f"   จำนวน predictions: {len(all_predictions)}")

# แสดงตัวอย่าง
import pandas as pd
df = pd.read_csv(OUTPUT_PATH)
print(f"\nตัวอย่าง submission (10 แถวแรก):")
print(df.head(10).to_string(index=False))
print(f"\nจำนวนแถวทั้งหมด: {len(df)}")
print(f"จำนวนค่าที่ไม่ใช่ 0: {(df['votes'] != 0).sum()}")

### ดาวน์โหลดไฟล์ (Google Colab)

In [ ]:
# ดาวน์โหลดไฟล์ submission (สำหรับ Google Colab)
try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
    print("📥 กำลังดาวน์โหลด submission.csv...")
except ImportError:
    print(f"ไฟล์ submission อยู่ที่: {OUTPUT_PATH}")